<a href="https://colab.research.google.com/github/kursatkara/MAE_5020_Spring_2026/blob/master/06_Notebook_0_hand_calc_one_epoch_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Notebook 0 — Verifying a One-Epoch Hand Calculation for a 2–3–2 Neural Network

## 🎯 Learning Objective

Use a **single forward pass, backward pass, and parameter update** for a small feedforward neural network to verify hand calculations performed in class.

This notebook is designed to match the in-class example exactly:

- **2 input neurons**
- **3 hidden neurons**
- **2 output neurons**
- prescribed initial weights and biases
- one forward pass
- one loss evaluation
- one backward pass
- one gradient-descent update

The goal is not to train a model well.  
The goal is to **trust the mechanics** of neural-network computation by matching the hand calculation carefully.

---

## 🧠 Challenge

Before using neural networks as general function approximators, we need to answer a simpler and more important question:

> If we specify the inputs, weights, biases, activation function, and learning rate, can we correctly compute one full training step?

That means verifying:

1. hidden-layer pre-activations,
2. hidden-layer activations,
3. output-layer pre-activations,
4. predicted outputs,
5. loss,
6. gradients,
7. updated weights and biases.

This notebook is intentionally small enough that every quantity can be inspected directly.



## 🔹 Network Structure

We consider a **2–3–2 neural network**, which means:

- **2 inputs**
- **3 hidden neurons**
- **2 outputs**

A simple diagram is:

```text
[x1, x2]  ──(W^1, b^1)──>  s^1  ──σ──>  a^1  ──(W^2, b^2)──>  s^2  ──σ──>  ŷ
```

Here:

- $x = [x_1, x_2]$ is the input row vector,
- $s^1$ is the hidden-layer pre-activation vector,
- $a^1$ is the hidden-layer activation vector,
- $s^2$ is the output-layer pre-activation vector,
- $\hat{y}$ is the predicted output vector.

We use the notation $s$ for the pre-activation values here to remain consistent with your in-class example.

---

## 🔹 Forward-pass equations

For the hidden layer:
$$
s^1 = xW^1 + b^1
$$
$$
a^1 = \sigma(s^1)
$$

For the output layer:
$$
s^2 = a^1 W^2 + b^2
$$
$$
\hat{y} = \sigma(s^2)
$$

---

## 🔹 Loss function

We compare the predicted output $\hat{y}$ to the target output $y$ using the mean squared error:
$$
L = \frac{1}{n_{\text{out}}}\sum_{j=1}^{n_{\text{out}}} \left(y_j - \hat{y}_j\right)^2
$$

Since there are 2 outputs in this notebook,
$$
L = \frac{1}{2}\left[(y_1-\hat{y}_1)^2 + (y_2-\hat{y}_2)^2\right].
$$

This matches the behavior of `torch.mean((y_true - y_pred)**2)` for a single sample with two outputs.

---

## 🔹 Big-picture interpretation

The full neural network is still a parameterized function:
$$
\hat{y} = f(x; \theta),
$$
where the trainable parameters are
$$
\theta = \{W^1, b^1, W^2, b^2\}.
$$

This notebook shows, in a fully concrete example, how one step of gradient-based learning changes those parameters.



## 🔹 Input, Target, Learning Rate, and Initial Parameters

We now define exactly the same values used in the in-class example.

### Input
$$
x = [0.85,\; 0.25]
$$

### Target output
$$
y = [1.0,\; 0.0]
$$

### Learning rate
$$
\eta = 0.3
$$

### Initial hidden-layer weights and biases
$$
W^1 =
\begin{bmatrix}
0.1 & 0.2 & 0.3 \\
0.4 & 0.5 & 0.6
\end{bmatrix},
\qquad
b^1 =
\begin{bmatrix}
0.0 & 0.0 & 0.0
\end{bmatrix}
$$

### Initial output-layer weights and biases
$$
W^2 =
\begin{bmatrix}
0.25 & 0.50 \\
0.10 & 0.20 \\
0.30 & 0.40
\end{bmatrix},
\qquad
b^2 =
\begin{bmatrix}
0.0 & 0.0
\end{bmatrix}
$$


In [1]:

import torch
import numpy as np

torch.set_printoptions(precision=6, sci_mode=False)

# Input values
i1, i2 = 0.85, 0.25
x = torch.tensor([[i1, i2]], dtype=torch.float32)

# Target output
y_true = torch.tensor([[1.0, 0.0]], dtype=torch.float32)

# Learning rate
lr = 0.3

# Initialize weights and biases
W1 = torch.tensor([[0.1, 0.2, 0.3],
                   [0.4, 0.5, 0.6]], dtype=torch.float32, requires_grad=True)
b1 = torch.tensor([[0.0, 0.0, 0.0]], dtype=torch.float32, requires_grad=True)

W2 = torch.tensor([[0.25, 0.50],
                   [0.10, 0.20],
                   [0.30, 0.40]], dtype=torch.float32, requires_grad=True)
b2 = torch.tensor([[0.0, 0.0]], dtype=torch.float32, requires_grad=True)

# Save copies of the initial values for later comparison
W1_old = W1.clone().detach()
b1_old = b1.clone().detach()
W2_old = W2.clone().detach()
b2_old = b2.clone().detach()

print("Input x:")
print(x)
print("\nTarget y_true:")
print(y_true)
print("\nInitial W1:")
print(W1_old)
print("\nInitial b1:")
print(b1_old)
print("\nInitial W2:")
print(W2_old)
print("\nInitial b2:")
print(b2_old)


Input x:
tensor([[0.850000, 0.250000]])

Target y_true:
tensor([[1., 0.]])

Initial W1:
tensor([[0.100000, 0.200000, 0.300000],
        [0.400000, 0.500000, 0.600000]])

Initial b1:
tensor([[0., 0., 0.]])

Initial W2:
tensor([[0.250000, 0.500000],
        [0.100000, 0.200000],
        [0.300000, 0.400000]])

Initial b2:
tensor([[0., 0.]])



## 🔹 Forward Pass

We now compute the forward pass exactly as done in class.

### Hidden layer
$$
s^1 = xW^1 + b^1
$$
$$
a^1 = \sigma(s^1)
$$

### Output layer
$$
s^2 = a^1 W^2 + b^2
$$
$$
\hat{y} = \sigma(s^2)
$$

At each step, print the intermediate values and compare them to the hand calculations.


In [2]:

# Forward pass
s1 = x @ W1 + b1
a1 = torch.sigmoid(s1)

s2 = a1 @ W2 + b2
y_pred = torch.sigmoid(s2)

print("Hidden-layer pre-activation s1:")
print(s1.detach())

print("\nHidden-layer activation a1:")
print(a1.detach())

print("\nOutput-layer pre-activation s2:")
print(s2.detach())

print("\nPredicted output y_pred:")
print(y_pred.detach())


Hidden-layer pre-activation s1:
tensor([[0.185000, 0.295000, 0.405000]])

Hidden-layer activation a1:
tensor([[0.546119, 0.573220, 0.599888]])

Output-layer pre-activation s2:
tensor([[0.373818, 0.627659]])

Predicted output y_pred:
tensor([[0.592381, 0.651958]])



## 🔹 Loss Evaluation

We compute the mean squared error:
$$
L = \frac{1}{2}\left[(y_1-\hat{y}_1)^2 + (y_2-\hat{y}_2)^2\right].
$$

In code, this is implemented as:

```python
loss = torch.mean((y_true - y_pred) ** 2)
```

Because there is one sample and two output values, this matches the expression above.


In [3]:

# Compute loss (mean squared error)
loss = torch.mean((y_true - y_pred) ** 2)

print("Loss:")
print(loss.item())


Loss:
0.29560136795043945



## 🔹 Backward Pass

We now compute gradients using backpropagation:
$$
\nabla_\theta L
$$

PyTorch applies the chain rule automatically when we call:

```python
loss.backward()
```

For verification, it is useful to inspect the gradients **before** the update step.

This is important pedagogically because students often focus only on the updated weights and miss the fact that the update comes from the gradients.


In [4]:

# Backpropagation
loss.backward()

print("Gradient dL/dW1:")
print(W1.grad)

print("\nGradient dL/db1:")
print(b1.grad)

print("\nGradient dL/dW2:")
print(W2.grad)

print("\nGradient dL/db2:")
print(b2.grad)


Gradient dL/dW1:
tensor([[0.010400, 0.004106, 0.006048],
        [0.003059, 0.001208, 0.001779]])

Gradient dL/db1:
tensor([[0.012235, 0.004830, 0.007116]])

Gradient dL/dW2:
tensor([[-0.053752,  0.080790],
        [-0.056420,  0.084799],
        [-0.059045,  0.088744]])

Gradient dL/db2:
tensor([[-0.098426,  0.147935]])



## 🔹 One Gradient-Descent Update

We now perform exactly one parameter update using:
$$
\theta^{\text{new}} = \theta^{\text{old}} - \eta \nabla_\theta L
$$

In this notebook, the learning rate is
$$
\eta = 0.3.
$$

After the update, we will compare the old and new parameter values directly.


In [5]:

# Update weights and biases
with torch.no_grad():
    W1 -= lr * W1.grad
    b1 -= lr * b1.grad
    W2 -= lr * W2.grad
    b2 -= lr * b2.grad

# Save gradients for optional discussion before clearing them
grad_W1 = W1.grad.clone()
grad_b1 = b1.grad.clone()
grad_W2 = W2.grad.clone()
grad_b2 = b2.grad.clone()

# Clear gradients
W1.grad.zero_()
b1.grad.zero_()
W2.grad.zero_()
b2.grad.zero_()


tensor([[0., 0.]])


## 🔹 Compare Old and New Parameters

This is the key verification stage.

Compare the printed values below to the hand calculations from class.

Questions to ask:

1. Do the updated weights match the hand-computed values?
2. Are the signs of the updates consistent with the gradients?
3. Which parameters changed the most, and why?


In [6]:

print("W1 (old):")
print(W1_old)
print("\nW1 (new):")
print(W1)

print("\nb1 (old):")
print(b1_old)
print("\nb1 (new):")
print(b1)

print("\nW2 (old):")
print(W2_old)
print("\nW2 (new):")
print(W2)

print("\nb2 (old):")
print(b2_old)
print("\nb2 (new):")
print(b2)


W1 (old):
tensor([[0.100000, 0.200000, 0.300000],
        [0.400000, 0.500000, 0.600000]])

W1 (new):
tensor([[0.096880, 0.198768, 0.298185],
        [0.399082, 0.499638, 0.599466]], requires_grad=True)

b1 (old):
tensor([[0., 0., 0.]])

b1 (new):
tensor([[-0.003671, -0.001449, -0.002135]], requires_grad=True)

W2 (old):
tensor([[0.250000, 0.500000],
        [0.100000, 0.200000],
        [0.300000, 0.400000]])

W2 (new):
tensor([[0.266126, 0.475763],
        [0.116926, 0.174560],
        [0.317713, 0.373377]], requires_grad=True)

b2 (old):
tensor([[0., 0.]])

b2 (new):
tensor([[ 0.029528, -0.044380]], requires_grad=True)



## 🔹 Parameter Change Summary

Sometimes it is useful to inspect not only the old and new values, but also the actual parameter changes:
$$
\Delta W = W^{\text{new}} - W^{\text{old}}.
$$

This makes it easier to connect:

- gradient sign,
- update direction,
- learning-rate scaling.


In [7]:

print("Delta W1 = W1_new - W1_old:")
print(W1 - W1_old)

print("\nDelta b1 = b1_new - b1_old:")
print(b1 - b1_old)

print("\nDelta W2 = W2_new - W2_old:")
print(W2 - W2_old)

print("\nDelta b2 = b2_new - b2_old:")
print(b2 - b2_old)


Delta W1 = W1_new - W1_old:
tensor([[-0.003120, -0.001232, -0.001815],
        [-0.000918, -0.000362, -0.000534]], grad_fn=<SubBackward0>)

Delta b1 = b1_new - b1_old:
tensor([[-0.003671, -0.001449, -0.002135]], grad_fn=<SubBackward0>)

Delta W2 = W2_new - W2_old:
tensor([[ 0.016126, -0.024237],
        [ 0.016926, -0.025440],
        [ 0.017713, -0.026623]], grad_fn=<SubBackward0>)

Delta b2 = b2_new - b2_old:
tensor([[ 0.029528, -0.044380]], grad_fn=<SubBackward0>)



## 🔍 Sanity Check

Before concluding that the one-epoch computation is correct, verify the following:

- Do the dimensions make sense?
  - $x$ is $(1 \times 2)$
  - $W^1$ is $(2 \times 3)$
  - $s^1$ and $a^1$ are $(1 \times 3)$
  - $W^2$ is $(3 \times 2)$
  - $s^2$ and $\hat{y}$ are $(1 \times 2)$

- Is the loss positive?
- Are the gradients nonzero?
- Are the updated parameters different from the initial ones?
- Do the update directions agree with gradient descent?

These checks should become routine in all neural-network workflows.


In [8]:

print("Shapes:")
print("x.shape      =", tuple(x.shape))
print("W1.shape     =", tuple(W1.shape))
print("b1.shape     =", tuple(b1.shape))
print("s1.shape     =", tuple(s1.shape))
print("a1.shape     =", tuple(a1.shape))
print("W2.shape     =", tuple(W2.shape))
print("b2.shape     =", tuple(b2.shape))
print("s2.shape     =", tuple(s2.shape))
print("y_pred.shape =", tuple(y_pred.shape))


Shapes:
x.shape      = (1, 2)
W1.shape     = (2, 3)
b1.shape     = (1, 3)
s1.shape     = (1, 3)
a1.shape     = (1, 3)
W2.shape     = (3, 2)
b2.shape     = (1, 2)
s2.shape     = (1, 2)
y_pred.shape = (1, 2)



## 🔹 Optional Verification by Recomputing the Forward Pass

After one gradient update, the prediction should typically move somewhat toward the target.

We can verify that by running one more forward pass using the updated parameters and comparing the new prediction and loss to the original ones.


In [9]:

with torch.no_grad():
    s1_new = x @ W1 + b1
    a1_new = torch.sigmoid(s1_new)
    s2_new = a1_new @ W2 + b2
    y_pred_new = torch.sigmoid(s2_new)
    loss_new = torch.mean((y_true - y_pred_new) ** 2)

print("Original prediction:")
print(y_pred.detach())

print("\nUpdated prediction after one gradient step:")
print(y_pred_new)

print("\nOriginal loss:")
print(loss.item())

print("\nUpdated loss after one gradient step:")
print(loss_new.item())


Original prediction:
tensor([[0.592381, 0.651958]])

Updated prediction after one gradient step:
tensor([[0.606276, 0.631408]])

Original loss:
0.29560136795043945

Updated loss after one gradient step:
0.2768476903438568



## 🔹 Reflection Questions

1. Why is this called a 2–3–2 neural network?
2. Why do we apply the sigmoid function at both the hidden and output layers in this example?
3. Why does `loss.backward()` produce gradients for all trainable parameters automatically?
4. Which parameter block appears most sensitive in this example: $W^1$, $b^1$, $W^2$, or $b^2$?
5. After one update, did the loss decrease? If so, why is that reasonable?

These questions help move students from arithmetic to interpretation.



## 🧠 Engineering Takeaway

This notebook verifies the full mechanics of one neural-network training step for a small multi-output system.

Even though the network is tiny, it already contains the essential ingredients of modern neural-network training:

- matrix-based forward propagation,
- nonlinear activation functions,
- a loss function,
- gradient computation through backpropagation,
- gradient-based parameter updates.

Once these mechanics are trusted in a small example, it becomes much easier to understand larger neural networks used in practice.



## 🚀 Next

After this notebook, a natural next step is to study what happens when the same update is repeated over many epochs.

That moves us from **verifying one training step** to **analyzing learning dynamics over time**.
